In [1]:
import sys
from pathlib import Path
for _candidate in [Path().resolve().parent / 'src', Path().resolve() / 'src']:
    if _candidate.exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        break

In [2]:
import os
import time
import gc
import pickle
gc.collect()
from encodec_compress_vs2 import EncodecCompression
from settings import Config
from config_species import get_settings
from preprocess import Preprocessing

c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
TARGET_SPECIES = "Gibbon" # Can be Thyolo, PTW, Gibbon
method_compression = 'encodec'
parameter_compression = '6.0' # Bandwidth
block_duration_sec= 30  #in sec

In [4]:
settings = get_settings(TARGET_SPECIES.lower())
config = Config(settings)

In [5]:

print(f"\n{'='*40}\nProcessing Species: {TARGET_SPECIES}\n{'='*40}")
species_folder = config.data.species_folder
folder_audio = "C:/Users/loren/Documents/Postdoc/Compressed_sensing/Data/Gibbon/Audio"
folder_compress = "C:/Users/loren/Documents/Postdoc/Compressed_sensing/Data/Gibbon/Compressed_Audio"



Processing Species: Gibbon


In [7]:
encodec = EncodecCompression(folder_audio, folder_compress, parameter_compression=parameter_compression, block_duration_sec=block_duration_sec)


c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [9]:
import torch
batch = torch.randn(
    8,
    encodec.model.channels,
    24000,
)

encoded = encodec.model.encode(batch)

print(len(encoded))
print(encoded[0][0].shape)

1
torch.Size([8, 8, 75])


In [6]:
print("\n--- 1. Running EnCodec Compression ---")
encodec = EncodecCompression(folder_audio, folder_compress, parameter_compression=parameter_compression, block_duration_sec=block_duration_sec)
starting = time.time()
times = encodec.compress()
execution_time_baseline = time.time() - starting
print(f"Compression execution time: {execution_time_baseline:.2f} seconds")


--- 1. Running EnCodec Compression ---

Compressing: HGSM3AB_0+1_20160303_060100.wav


c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Encode full file: 121.76763534545898

Compressing: HGSM3AB_0+1_20160304_060000.wav


KeyboardInterrupt: 

In [ ]:


print(f"\n{'='*40}\nProcessing Species: {TARGET_SPECIES}\n{'='*40}")
species_folder = config.data.species_folder
folder_audio = os.path.join(species_folder, 'Audio')
folder_compress = os.path.join(species_folder, 'Compressed_Audio')

print("\n--- 1. Running EnCodec Compression ---")
compression = EncodecCompression(folder_audio, folder_compress, parameter_compression=parameter_compression)
starting = time.time()
times = compression.compress()
execution_time_baseline = time.time() - starting
print(f"Compression execution time: {execution_time_baseline:.2f} seconds")

print("\n--- 2. Dataset Processing ---")
preprocess = Preprocessing(
    **config.preprocessing.dict(),
    species_folder=species_folder,
    positive_class=config.data.positive_class,
    negative_class=config.data.negative_class,
)

saved_data_folder = os.path.join(species_folder, 'Saved_Data')
os.makedirs(saved_data_folder, exist_ok=True)

types = ['train', 'val', 'test']
for dataset_type in types:
    print(f"\nProcessing {dataset_type} split...")
    is_train = (dataset_type == 'train')
    
    X_calls, Y_calls = preprocess.create_dataset(
        dataset=dataset_type, 
        method_compression=method_compression, 
        parameter_compression=parameter_compression, 
        preprocessing=True, 
        data_augmentation=is_train, 
        noise_reduction=False
    )
    Y = preprocess._one_hot_encode(Y_calls)
    
    print(f"\n--- 3. Saving Extracted Dataset ({dataset_type}) ---")
    with open(Path(saved_data_folder, f"{config.data.positive_class}_X_{dataset_type}_{method_compression}_{parameter_compression}.pkl"), 'wb') as f:
        pickle.dump(X_calls, f)
    with open(Path(saved_data_folder, f"{config.data.positive_class}_Y_{dataset_type}.pkl"), 'wb') as f:
        pickle.dump(Y, f)
    print(f"Finished processing {dataset_type} split!\n")
    del X_calls, Y_calls, Y
    gc.collect()

print('Finished end-to-end pipeline evaluation for Encodec!')


Processing Species: PTW

--- 1. Running EnCodec Compression ---


/home/milanto/.local/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


compression file : 20210125_140315.WAV


/home/milanto/.local/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/milanto/.local/lib/python3.9/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more informa

compression file : 20210124_182420.WAV
compression file : 20210122_164355.WAV
compression file : 20210121_184425.WAV
compression file : 20210124_040045.WAV
compression file : 20210122_144325.WAV
compression file : 20210124_144325.WAV
compression file : 20210123_084155.WAV
compression file : 20210125_030030.WAV
compression file : 20210125_142320.WAV
compression file : 20210125_034040.WAV
compression file : 20210125_160345.WAV
compression file : 20210121_154340.WAV
compression file : 20210125_042050.WAV
compression file : 20210124_034040.WAV
compression file : 20210123_074140.WAV
compression file : 20210122_134310.WAV
compression file : 20210121_142320.WAV
compression file : 20210124_142320.WAV
compression file : 20210124_130300.WAV
compression file : 20210125_184425.WAV
compression file : 20210124_080145.WAV
compression file : 20210122_172405.WAV
compression file : 20210124_154340.WAV
compression file : 20210125_050100.WAV
compression file : 20210123_160345.WAV
compression file : 202101